<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# Inspect a Local ROS 2 Graph

The previous notebook built the supplied package and confirmed that ROS 2 could find its executables. The next step is to start the publisher and investigate how its running behaviour appears in the graph.

Which node does the executable create? Is anything subscribed to its topic, and do messages arrive at the expected rate? What else does the node expose, and can its behaviour be changed without restarting it?

This notebook connects the publisher's terminal output to its discovered interfaces, then observes how subscribing changes the graph, measures the arriving messages, and changes a parameter of the running node.

## Start the installed publisher

The commands run in the editor terminals, using the package built in the previous notebook, with a router reachable (notebook 03). Earlier exercise publishers should be stopped so their endpoints do not contribute to the observations.

In Terminal 1:

```bash
ros2 run ros2_basics my_publisher_node
```

The publisher should begin logging messages such as:

```shell
[INFO] [1793450001.123456789] [my_publisher_node]: publishing: "Hello from ROS 2! [0]"
[INFO] [1793450002.123456789] [my_publisher_node]: publishing: "Hello from ROS 2! [1]"
```

The time stamps differ, while the text should remain `Hello from ROS 2!` followed by a counter. The lines continue roughly once per second, showing that the publisher's callback is executing. Leave Terminal 1 running while Terminal 2 performs the inspection.

## Find the node and its topic

In Terminal 2, listing the nodes should reveal the publisher:

```bash
ros2 node list
```

```shell
/my_publisher_node
```

If the listing remains empty while publication continues, the direct-discovery check introduced in notebook 04 (`--no-daemon --spin-time 5`) helps distinguish an observation problem from a stopped publisher.

The topic listing identifies the interfaces visible alongside that node:

```bash
ros2 topic list -t
```

With only the exercise publisher running, the result is:

```shell
/chatter [std_msgs/msg/String]
/parameter_events [rcl_interfaces/msg/ParameterEvent]
/rosout [rcl_interfaces/msg/Log]
```

The application publishes text on `/chatter`, while the other entries support parameter events and logging. To find out how many endpoints currently use `/chatter`, inspect the topic:

```bash
ros2 topic info /chatter
```

```shell
Type: std_msgs/msg/String
Publisher count: 1
Subscription count: 0
```

One publisher is present, and no subscriptions have joined yet. Publishing can continue in this state because the publisher does not require a subscriber before producing messages.

## Observe messages and the subscription they create

The previous notebooks used `ros2 topic echo` to receive messages. Keeping that observer running now makes its subscription visible to another inspection command. In Terminal 2:

```bash
ros2 topic echo /chatter std_msgs/msg/String
```

The explicit type matches the topic listing and allows the observer to create its subscription without first inferring the message type from discovery. Messages should begin arriving:

```shell
data: Hello from ROS 2! [12]
---
data: Hello from ROS 2! [13]
---
```

Each `---` ends one displayed message. The command continues receiving, so Terminal 2 remains occupied until `Ctrl-C` stops it. To inspect the graph while both publisher and observer remain active, open Terminal 3:

```bash
ros2 topic info /chatter
```

The topic information should now include one subscription:

```shell
Type: std_msgs/msg/String
Publisher count: 1
Subscription count: 1
```

The observer has become part of the graph it inspects. Its displayed messages establish that this subscription is receiving the publisher's data. Stop the echo command with `Ctrl-C` in Terminal 2, then repeat the topic inspection in Terminal 3: after discovery reflects the change, the subscription count returns to `0`. Terminal 1 continues publishing throughout this change.

## Connect the endpoint to its delivery settings

The brief topic inspection reports counts of publishers and subscribers. Its verbose form adds information about each endpoint, including the node that owns it and its quality-of-service settings:

```bash
ros2 topic info --verbose /chatter
```

With the echo command stopped, selected lines from the publisher endpoint are:

```shell
Node name: my_publisher_node
Node namespace: /
Topic type: std_msgs/msg/String
Endpoint type: PUBLISHER
QoS profile:
  Reliability: RELIABLE
  History (Depth): KEEP_LAST (10)
  Durability: VOLATILE
```

The node name connects this endpoint to `/my_publisher_node`. The QoS fields describe its delivery policies: `RELIABLE` means every message is delivered (retransmitted if needed), `KEEP_LAST (10)` is the queue depth written in the code (`create_publisher(String, "chatter", 10)`), and `VOLATILE` means a subscriber that joins later does not receive the messages published before it joined. The [ROS 2 QoS documentation](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Quality-of-Service-Settings.html) explains these policies and their compatibility rules; notebook 10 compares them with the policies of the robot's sensors.

## Measure the arriving message rate

Receiving one message establishes delivery, but an application may also depend on how regularly new messages arrive. In Terminal 2:

```bash
ros2 topic hz /chatter
```

This command creates a subscription and estimates the rate from received messages. After a few received samples, it reports the observed arrival rate:

```shell
average rate: 1.000
	min: 0.999s max: 1.001s std dev: 0.00052s window: 5
```

An average near `1.000` corresponds to approximately one received message per second. The minimum and maximum values describe observed intervals between messages, while the standard deviation summarises their variation. These measurements describe reception by this observer, so scheduling and system load can affect them. Stop the measurement with `Ctrl-C` before continuing.

### A note on latency

Alongside arrival rate, a robotics application needs to consider how old its data is when it becomes available for processing. Messages can arrive regularly while carrying information acquired earlier. A time stamp in the message allows an observer to measure that age. The `String` messages on `/chatter` contain only text; most robot messages carry a `std_msgs/Header` with a `stamp` (notebook 03 showed it in `Twist2DStamped` and `WheelEncoderStamped`). A small stamped example provides the field. In Terminal 3:

```bash
ros2 topic pub --rate 1 --node-name stamped_publisher /latency_demo geometry_msgs/msg/PointStamped "{header: {stamp: now, frame_id: map}, point: {x: 1.0, y: 0.0, z: 0.0}}"
```

The special value `now` tells the command-line publisher to refresh the time stamp for each publication. Leave it running and, in Terminal 2, measure the elapsed time from that time stamp to the receiving callback:

```bash
ros2 topic delay /latency_demo
```

```shell
average delay: 0.001
	min: 0.000s max: 0.001s std dev: 0.00006s window: 8
```

The reported durations are in seconds: about one millisecond here, since both processes run in the same container. The tool calculates `delay = time at the receiving callback - header stamp`. For sensor data, the stamp identifies the acquisition time, in which case the measured age also includes the processing before publication. Across computers (your laptop and the robot) clock synchronisation becomes necessary, because an offset between the clocks also appears in the subtraction.

Stop the delay measurement in Terminal 2 and the stamped publisher in Terminal 3 with `Ctrl-C`. Leave `/my_publisher_node` in Terminal 1 running.

## Inspect the node's other interfaces

Topic inspection starts from a stream. To examine the participating node as a whole:

```bash
ros2 node info /my_publisher_node
```

The output lists every interface of the node. The publisher section includes:

```shell
  Publishers:
    /chatter: std_msgs/msg/String
    /parameter_events: rcl_interfaces/msg/ParameterEvent
    /rosout: rcl_interfaces/msg/Log
  Service Servers:
    /my_publisher_node/describe_parameters: rcl_interfaces/srv/DescribeParameters
    /my_publisher_node/get_parameters: rcl_interfaces/srv/GetParameters
    /my_publisher_node/set_parameters: rcl_interfaces/srv/SetParameters
    ...
```

Every node offers these parameter services. They are what the `ros2 param` tool (and the sliders of the notebooks) talk to. The available parameter names can be listed:

```bash
ros2 param list /my_publisher_node
```

```shell
  message
  rate_hz
  start_type_description_service
  use_sim_time
```

`message` and `rate_hz` were declared by the node (the two others are standard). Read one:

```bash
ros2 param get /my_publisher_node rate_hz
```

```shell
Double value is: 1.0
```

A Python constant becomes remotely configurable only if the implementation exposes it as a parameter, which is what `my_publisher_node` does.

## Change the behaviour of the running node

Set the publication rate and the text, without stopping the node:

```bash
ros2 param set /my_publisher_node rate_hz 2.0
ros2 param set /my_publisher_node message "Hello from the terminal"
```

Each command answers `Set parameter successful`. Terminal 1 logs `parameter rate_hz = 2.0` and the new text, and now prints two lines per second. Confirm the new rate from Terminal 2:

```bash
ros2 topic hz /chatter
```

```shell
average rate: 2.000
	min: 0.499s max: 0.501s std dev: 0.00041s window: 10
```

A parameter change applies to the running instance only. Restarting the node restores the defaults from its code or from the yaml file given to its launch file; notebook 09 looks at how the node declares the parameters and reacts to changes.

Finally:

```bash
ros2 action list -t
```

With only this exercise running, the command returns an empty listing because the publisher exposes no action interface.

## Finish the observation

With the echo and rate-measurement commands stopped, stop the publisher in Terminal 1 using `Ctrl-C`. A direct discovery check allows time for the graph to reflect its departure:

```bash
ros2 node list --no-daemon --spin-time 5
```

With only the exercise nodes previously running, the listing should now be empty.

Starting the installed executable created `/my_publisher_node` and its `/chatter` publisher. Echoing the topic added a visible subscription and revealed its messages, while rate measurement established how regularly those messages arrived. The stamped-message example measured message age at the observer, showing why time stamps and a shared time basis are needed to interpret delay. Node inspection connected those observations to delivery settings and to the parameters of the node, which were changed while it ran. The next notebook applies the same tools to the Duckiebot.

## Further reading

The ROS 2 Jazzy [CLI tutorials](https://docs.ros.org/en/jazzy/Tutorials/Beginner-CLI-Tools.html) cover graph inspection and parameters, while the [QoS documentation](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Quality-of-Service-Settings.html) explains delivery policies.

Let's move to [the next notebook about the Duckiebot's graph](../notebooks/08_inspect_the_duckiebot_graph.ipynb).